This notebook requires a `tRNAscan-SE` output file.

In [ ]:
INPUT_PATH = ""
SAMPLES_PER_GROUP = 500

In [ ]:
from collections import defaultdict
from pathlib import Path
import re

import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"],
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
})

LABEL_MAPPING = {
    "Same codon": "Exact match",
    "Wobble pair": "Wobble pairing",
    "Other same type": "Isoacceptor",
    "Wrong Type": "Incorrect isotype",
    "Invalid/Missing": "No detection",
}

PROFESSIONAL_ORDER = [
    "Exact match",
    "Wobble pairing",
    "Isoacceptor",
    "Incorrect isotype",
    "No detection",
]

CATEGORY_COLORS_PRO = {
    "Exact match": "#2ca02c",
    "Wobble pairing": "#1f77b4",
    "Isoacceptor": "#ff7f0e",
    "Incorrect isotype": "#7f7f7f",
    "No detection": "#d62728",
}

NATURE_COLORS = {
    "Exact match": "#4393C3",
    "Wobble pairing": "#92C5DE",
    "Isoacceptor": "#F4A582",
    "Incorrect isotype": "#BABABA",
    "No detection": "#D6604D",
}

In [ ]:
def set_nature_style() -> None:
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica"],
        "font.size": 14,
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 14,
        "axes.linewidth": 1.0,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 5.0,
        "ytick.major.size": 5.0,
    })


def _smart_split(line: str):
    if "\t" in line:
        return [col.strip() for col in line.split("\t")]
    return [part.strip() for part in re.split(r"\s{2,}", line.strip())]


def _collapse_name(name: str) -> str:
    s = str(name).strip()
    s = re.sub(r"[ \t]+$", "", s)
    match = re.match(r"^(.*?)(?:[_\s-]\s*\d+)?$", s)
    return match.group(1).rstrip("_- ") if match else s


def _parse_pred_from_groupname(groupname: str):
    s = str(groupname).lower()
    codon_match = re.search(r"trna[a-z]*-([acgtu]{3})", s)
    pred_codon = codon_match.group(1).upper().replace("T", "U") if codon_match else None
    type_matches = list(re.finditer(r"trna-([a-z]+)", s))
    pred_type = type_matches[-1].group(1).lower() if type_matches else None
    return pred_codon, pred_type


def read_trnascan_output(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    lines = path.read_text(encoding="utf-8").splitlines()
    lines = [line.rstrip("\n") for line in lines]
    lines = [line for line in lines if line.strip() and not set(line.strip()) <= set("-")]

    header_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^Name(\s|$)", line):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError("Could not find a header line starting with 'Name'.")

    header_cols = _smart_split(lines[header_idx])
    data_lines = lines[header_idx + 1:]

    norm_cols = []
    dup_count = defaultdict(int)
    for col in header_cols:
        base = col.strip()
        if base in {"Begin", "End"}:
            dup_count[base] += 1
            base = f"{base}{dup_count[base]}"
        elif base == "tRNA #":
            base = "tRNA_num"
        elif base not in {"Name", "Type", "Codon", "Score", "Note"}:
            base = base.replace(" ", "_")
        norm_cols.append(base)

    rows = []
    for line in data_lines:
        if re.match(r"^Name(\s|$)", line):
            break
        parts = _smart_split(line)
        if len(parts) < len(norm_cols):
            parts = parts + [""] * (len(norm_cols) - len(parts))
        rows.append(parts[:len(norm_cols)])

    df = pd.DataFrame(rows, columns=norm_cols)

    if "Name" in df.columns and not df.empty:
        df = df.drop_duplicates(subset=["Name"], keep="first").reset_index(drop=True)

    for col in ["tRNA_num", "Begin1", "End1", "Begin2", "End2", "Score"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Begin1" in df.columns and "End1" in df.columns:
        df["Length"] = abs(df["End1"] - df["Begin1"]) + 1

    if "Codon" not in df.columns:
        df["Codon"] = ""

    df["Codon"] = df["Codon"].astype(str).str.upper().str.replace("[^ACGTU]", "", regex=True)
    df["Codon_mRNA"] = df["Codon"].str.replace("T", "U")
    df["GroupName"] = df["Name"].apply(_collapse_name)

    preds = df[["GroupName"]].drop_duplicates().copy()
    preds[["pred_codon", "pred_type"]] = preds["GroupName"].apply(
        lambda g: pd.Series(_parse_pred_from_groupname(g))
    )
    df = df.merge(preds, on="GroupName", how="left")
    return df


def categorize_predictions(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["Type_norm"] = d["Type"].astype(str).str.lower()
    d["pred_type_norm"] = d["pred_type"].astype(str).str.lower()
    d["Codon_norm"] = d["Codon_mRNA"].astype(str).str.upper()
    d["pred_codon_norm"] = d["pred_codon"].astype(str).str.upper()

    same_type = d["Type_norm"] == d["pred_type_norm"]
    same_codon_basic = d["Codon_norm"] == d["pred_codon_norm"]

    is_cau = d["Codon_norm"] == "CAU"
    is_imet = d["Type_norm"] == "imet"
    same_codon = same_codon_basic | (is_cau & is_imet)

    len3_both = (d["Codon_norm"].str.len() == 3) & (d["pred_codon_norm"].str.len() == 3)
    same_tail = d["Codon_norm"].str[1:] == d["pred_codon_norm"].str[1:]
    wobble_pair = same_type & (~same_codon_basic) & len3_both & same_tail
    other_same_type = same_type & ~same_codon & ~wobble_pair

    conditions = [same_codon, wobble_pair, other_same_type]
    choices = ["Same codon", "Wobble pair", "Other same type"]
    d["Category"] = np.select(conditions, choices, default="Wrong Type")
    return d

In [ ]:
try:
    print(f"Reading data from {INPUT_PATH}...")
    df_raw = read_trnascan_output(INPUT_PATH)
    print(f"Successfully loaded {len(df_raw)} predicted records")
    print("Applying classification logic...")
    df_labeled = categorize_predictions(df_raw)
    print("Data processing complete. You can now run the plotting cells below.")
except Exception as e:
    print(f"Data loading or processing failed: {e}")

In [ ]:
def plot_fidelity_main_optimized(df_labeled: pd.DataFrame, samples_per_group: int = 500) -> None:
    set_nature_style()

    stats = df_labeled.groupby(["pred_type_norm", "Category"]).size().unstack(fill_value=0)
    group_counts = df_labeled.groupby("pred_type_norm")["GroupName"].nunique()

    stats["Total_Expected"] = group_counts * samples_per_group
    detected_cols = [c for c in stats.columns if c != "Total_Expected"]
    stats["Detected_Total"] = stats[detected_cols].sum(axis=1)
    stats["Invalid/Missing"] = stats["Total_Expected"] - stats["Detected_Total"]

    old_cols = ["Same codon", "Wobble pair", "Other same type", "Wrong Type", "Invalid/Missing"]
    for col in old_cols:
        if col not in stats.columns:
            stats[col] = 0

    stats_pct = stats[old_cols].div(stats["Total_Expected"], axis=0) * 100
    stats_pct = stats_pct.sort_values("Same codon", ascending=False)
    stats_pct = stats_pct.rename(columns=LABEL_MAPPING)
    stats_pct = stats_pct[PROFESSIONAL_ORDER]

    fig, ax = plt.subplots(figsize=(10.0, 5.5))
    stats_pct.plot(
        kind="bar",
        stacked=True,
        color=[NATURE_COLORS.get(c) for c in PROFESSIONAL_ORDER],
        ax=ax,
        width=0.8,
        edgecolor="black",
        linewidth=0.2,
    )

    ax.set_ylabel("Percentage (%)", fontsize=18)
    ax.set_xlabel("")
    ax.set_xticklabels(
        [label.get_text().capitalize() for label in ax.get_xticklabels()],
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.legend(
        bbox_to_anchor=(0.5, 1.05),
        loc="lower center",
        ncol=5,
        frameon=False,
        handletextpad=0.4,
        columnspacing=1.2,
        handlelength=1.2,
    )
    sns.despine()
    plt.tight_layout()
    plt.show()


plot_fidelity_main_optimized(df_labeled, SAMPLES_PER_GROUP)

In [ ]:
def plot_score_distribution_main_optimized(df_labeled: pd.DataFrame) -> None:
    set_nature_style()

    plot_df = df_labeled[(df_labeled["Score"] > 0) & (df_labeled["Category"] != "Invalid/Missing")].copy()
    plot_df["Category_Pro"] = plot_df["Category"].map(LABEL_MAPPING)
    order_pro = [
        c for c in PROFESSIONAL_ORDER
        if c in plot_df["Category_Pro"].unique() and c != "No detection"
    ]
    palette_colors = [NATURE_COLORS.get(c, "#333333") for c in order_pro]

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.axhline(y=70, color="#888888", linestyle="--", linewidth=1.0, zorder=0)

    sns.violinplot(
        data=plot_df,
        x="Category_Pro",
        y="Score",
        order=order_pro,
        palette=palette_colors,
        inner=None,
        linewidth=0.8,
        ax=ax,
        cut=0,
        zorder=1,
    )

    for collection in ax.collections:
        if isinstance(collection, mpl.collections.PolyCollection):
            collection.set_alpha(0.3)

    sns.stripplot(
        data=plot_df,
        x="Category_Pro",
        y="Score",
        order=order_pro,
        palette=palette_colors,
        size=1.5,
        jitter=0.15,
        alpha=0.4,
        ax=ax,
        edgecolor="none",
        zorder=2,
    )

    medians = plot_df.groupby("Category_Pro")["Score"].median()
    for i, cat in enumerate(order_pro):
        if cat in medians:
            ax.hlines(
                y=medians[cat],
                xmin=i - 0.25,
                xmax=i + 0.25,
                color="black",
                linewidth=2.0,
                zorder=3,
            )

    ax.set_ylabel("tRNAscan-SE score (bits)", fontsize=18)
    ax.set_xlabel("")
    ax.set_xticklabels([label.get_text().capitalize() for label in ax.get_xticklabels()])
    sns.despine()
    plt.tight_layout()
    plt.show()


plot_score_distribution_main_optimized(df_labeled)

In [ ]:
def plot_supplementary_anticodon_fidelity(df_labeled: pd.DataFrame, samples_per_group: int = 500) -> None:
    plot_df = df_labeled.copy()
    plot_df["Target_Full"] = plot_df["pred_type_norm"].str.capitalize() + "-" + plot_df["pred_codon_norm"]

    stats = plot_df.groupby(["Target_Full", "Category"]).size().unstack(fill_value=0)
    stats["Total_Expected"] = samples_per_group
    valid_cols = [c for c in stats.columns if c != "Total_Expected"]
    stats["Detected_Sum"] = stats[valid_cols].sum(axis=1)
    stats["Invalid/Missing"] = stats["Total_Expected"] - stats["Detected_Sum"]
    stats["Invalid/Missing"] = stats["Invalid/Missing"].clip(lower=0)

    old_cols = ["Same codon", "Wobble pair", "Other same type", "Wrong Type", "Invalid/Missing"]
    for col in old_cols:
        if col not in stats.columns:
            stats[col] = 0

    stats_pct = stats[old_cols].div(stats["Total_Expected"], axis=0) * 100
    stats_pct["AA_Sort"] = stats_pct.index.str.split("-").str[0]
    stats_pct = stats_pct.sort_values(["AA_Sort", "Same codon"], ascending=[True, False])
    stats_pct = stats_pct.drop(columns="AA_Sort")
    stats_pct = stats_pct.rename(columns=LABEL_MAPPING)
    stats_pct = stats_pct[PROFESSIONAL_ORDER]

    fig_height = len(stats_pct) * 0.2 + 2.0
    fig, ax = plt.subplots(figsize=(7, fig_height))
    stats_pct.plot(
        kind="barh",
        stacked=True,
        color=[CATEGORY_COLORS_PRO.get(c) for c in PROFESSIONAL_ORDER],
        ax=ax,
        width=0.8,
        edgecolor="none",
    )

    ax.set_xlabel("Percentage of Sequences (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)
    ax.tick_params(axis="y", labelsize=7)
    ax.invert_yaxis()
    ax.legend(bbox_to_anchor=(0.5, -0.04), loc="upper center", ncol=3, frameon=False, fontsize=7)
    sns.despine(bottom=True)
    ax.xaxis.grid(True, linestyle="--", alpha=0.5)
    plt.subplots_adjust(bottom=0.1)
    plt.show()


plot_supplementary_anticodon_fidelity(df_labeled, SAMPLES_PER_GROUP)

In [ ]:
def quick_check_length_vloop_optimized(df_labeled: pd.DataFrame) -> None:
    set_nature_style()

    plot_df = df_labeled.copy()
    if "Length" not in plot_df.columns:
        if "Begin1" in plot_df.columns and "End1" in plot_df.columns:
            plot_df["Length"] = abs(plot_df["End1"] - plot_df["Begin1"]) + 1
        else:
            print("Length information is not available. Skipping plot.")
            return

    plot_df["Is_Long_Vloop_AA"] = plot_df["pred_type_norm"].isin(["leu", "ser"])

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.histplot(
        data=plot_df,
        x="Length",
        hue="Is_Long_Vloop_AA",
        bins=100,
        palette={True: "#e74c3c", False: "#7f7f7f"},
        element="step",
        fill=True,
        alpha=0.5,
        linewidth=1.5,
        stat="density",
        common_norm=False,
        ax=ax,
    )

    ax.set_xlabel("Sequence length (nt)", fontsize=18)
    ax.set_ylabel("Density", fontsize=18)
    ax.set_xlim(65, 100)

    red_patch = mpatches.Patch(color="#e74c3c", label="Class II (Leu/Ser)", alpha=0.6)
    grey_patch = mpatches.Patch(color="#7f7f7f", label="Class I (Others)", alpha=0.6)
    ax.legend(handles=[red_patch, grey_patch], frameon=False, loc="upper right", fontsize=16)

    sns.despine()
    plt.tight_layout()
    plt.show()


quick_check_length_vloop_optimized(df_labeled)